# Demo: inferensi foto baru

Foto yang diambil saat sesi dimasukkan ke `demo/photos/` (atau path-nya ditulis di `PHOTOS`), lalu model hasil training
dipakai untuk segmentasi instance. Kode load model dan gambar mask **sama persis** dengan `evaluation/evaluate_models.ipynb`
(predictor dari kode training Kaggle), jadi hasilnya konsisten dengan angka evaluasi.

**Perubahan cepat saat demo** (ubah di sel 1, jalankan sel 1, lalu sel di kolom kanan):

| Permintaan | Ubah | Jalankan ulang |
|---|---|---|
| ganti threshold skor | `CONF_THRESH` | sel 5 saja (tanpa inferensi ulang) |
| tampilkan kelas tertentu saja | `SHOW_CLASSES` | sel 5 |
| AP satu kelas | `AP_CLASS`, `AP_TEST_SET` | sel 6 |
| ganti model atau proporsi data latih | `MODEL`, `SPLIT` | sel 3, 4, 5 |
| foto baru | taruh di `demo/photos/` atau tambah ke `PHOTOS` | sel 4, 5 |

## 1. Placeholder

In [ ]:
PHOTOS = [
    "photos/",                       # semua foto di demo/photos/
    # "photos/foto_sesi_1.jpg",      # atau foto satu per satu
    # "../test_set/images/20261121_200651.jpg",
]
MODEL = "rfdetr_seg_l"               # "rfdetr_seg_l" | "yolo26l_seg" | "maskrcnn_r50_fpn"
SPLIT = 100                          # proporsi data latih (%): 25 | 50 | 100
CONF_THRESH = 0.5                    # skor minimum deteksi yang ditampilkan
SHOW_CLASSES = []                    # [] = semua kelas, contoh ["chair", "table"]
LONG_SIDE = 1280                     # sisi terpanjang foto sebelum inferensi (sama dengan test set sendiri)
AP_CLASS = "chair"                   # kelas yang AP-nya ditampilkan di sel 6
AP_TEST_SET = "own"                  # "own" (40 foto sendiri) | "original" (92 gambar test dataset)

## 2. Setup (pakai ulang kode `evaluate_models.ipynb`)

In [ ]:
import gc, json, time, warnings
from difflib import get_close_matches
from pathlib import Path

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)  # deprecation notice torch/rfdetr, tidak memengaruhi hasil

HERE = Path.cwd()  # folder demo/
ROOT = next(p for p in [HERE, *HERE.parents] if (p / "training" / "src").is_dir())

# sel evaluate_models.ipynb: setup + import (2), konfigurasi + path checkpoint (4), predictor (8), gambar mask (10)
_eval_nb = json.loads((ROOT / "evaluation" / "evaluate_models.ipynb").read_text())
for i in (2, 4, 8, 10):
    exec("".join(_eval_nb["cells"][i]["source"]))
assert callable(globals().get("load_predictor")) and callable(globals().get("draw_predictions")), \
    "urutan sel evaluate_models.ipynb berubah, sesuaikan indeks sel di atas"


def check_classes(names, what):
    bad = [n for n in names if n not in NAMES]
    assert not bad, f"{what} tidak dikenal: {bad}, mungkin maksudnya {[get_close_matches(n, NAMES, 3) for n in bad]}"


print(len(NAMES), "kelas:", ", ".join(NAMES))

## 3. Load model

In [ ]:
assert MODEL in MODELS, f"MODEL harus salah satu dari {MODELS}"
assert SPLIT in [C.pct(f) for f in SPLITS], f"SPLIT harus salah satu dari {[C.pct(f) for f in SPLITS]}"

predict = model = None  # lepas model sebelumnya dari GPU
gc.collect()
torch.cuda.empty_cache()

predict, model = load_predictor(MODEL, SPLIT / 100)
LOADED = (MODEL, SPLIT)
print(f"model {MODEL}, data latih {SPLIT}%, checkpoint {weights(MODEL, SPLIT / 100).relative_to(ROOT)}")

## 4. Inferensi

In [ ]:
def photo_paths(items):
    """Isi PHOTOS -> daftar file gambar. Folder diambil semua gambarnya."""
    paths = []
    for item in items:
        p = Path(item) if Path(item).is_absolute() else HERE / item
        assert p.exists(), f"tidak ditemukan: {p}"
        paths += sorted(q for q in p.iterdir() if q.suffix.lower() in C.IMG_EXTS) if p.is_dir() else [p]
    assert paths, "belum ada foto: taruh foto di demo/photos/ atau isi PHOTOS"
    return paths


def prepare(path):
    """Foto HP diputar sesuai EXIF dan diperkecil ke LONG_SIDE, sama seperti test set sendiri."""
    im = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    s = LONG_SIDE / max(im.size)
    if s < 1:
        im = im.resize((round(im.width * s), round(im.height * s)), Image.Resampling.BILINEAR)
    out = HERE / ".cache" / f"{path.stem}.jpg"
    out.parent.mkdir(exist_ok=True)
    im.save(out, quality=95)
    return out


assert globals().get("LOADED") == (MODEL, SPLIT), "MODEL/SPLIT berubah: jalankan sel 3 dulu"
PREDS = {}
for path in photo_paths(PHOTOS):
    img_path = prepare(path)
    t0 = time.perf_counter()
    with torch.no_grad():
        PREDS[img_path] = predict(img_path)  # semua deteksi skor >= 0.001, CONF_THRESH diterapkan di sel 5
    print(f"{path.name}: {len(PREDS[img_path])} deteksi, {time.perf_counter() - t0:.2f} s")
PRED_MODEL = LOADED

## 5. Hasil (threshold dan filter kelas)

In [ ]:
assert globals().get("PRED_MODEL") == (MODEL, SPLIT), "MODEL/SPLIT berubah: jalankan sel 3 dan 4 dulu"
check_classes(SHOW_CLASSES, "SHOW_CLASSES")

out_dir = HERE / "outputs" / f"{MODEL}_split{SPLIT}_conf{CONF_THRESH}"
out_dir.mkdir(parents=True, exist_ok=True)
for img_path, preds in PREDS.items():
    keep = [p for p in preds if p["score"] >= CONF_THRESH
            and (not SHOW_CLASSES or NAMES[p["category_id"] - 1] in SHOW_CLASSES)]
    drawn = draw_predictions(cv2.imread(str(img_path)), keep)
    cv2.imwrite(str(out_dir / img_path.name), drawn)

    plt.figure(figsize=(12, 9))
    plt.imshow(drawn[..., ::-1])  # BGR (OpenCV) -> RGB (matplotlib)
    plt.title(f"{img_path.name} | {MODEL} {SPLIT}% | skor >= {CONF_THRESH} | {len(keep)} objek")
    plt.axis("off")
    plt.show()
    rows = [{"kelas": NAMES[p["category_id"] - 1], "skor": round(p["score"], 3), "luas_px": int(p["mask"].sum())}
            for p in keep]
    display(pd.DataFrame(rows, columns=["kelas", "skor", "luas_px"]).sort_values("skor", ascending=False))
print("gambar tersimpan di", out_dir.relative_to(ROOT))

## 6. AP satu kelas

AP butuh ground truth, jadi diambil dari hasil evaluasi test set berlabel (`evaluation/results_{own,original}_test/`),
bukan dari foto sesi yang belum dilabel. AP = mask AP IoU 0.50:0.95, AP50 = IoU 0.50.

In [ ]:
check_classes([AP_CLASS], "AP_CLASS")
assert AP_TEST_SET in ("own", "original"), 'AP_TEST_SET harus "own" atau "original"'

rows = []
for m in MODELS:
    for f in SPLITS:
        pc = pd.read_csv(EVAL_DIR / f"results_{AP_TEST_SET}_test" / f"split_{C.pct(f)}" / m / "per_class_ap.csv")
        r = pc.loc[pc["name"] == AP_CLASS].iloc[0]
        rows.append({"model": m, "split": C.pct(f), "num_gt": int(r["num_gt"]), "AP": r["AP"], "AP50": r["AP50"]})
ap = pd.DataFrame(rows)

n_gt = ap["num_gt"].iloc[0]
print(f"kelas '{AP_CLASS}': {n_gt} instance di test set {AP_TEST_SET}")
if n_gt == 0:
    print("kelas ini tidak ada di test set ini, AP tidak terdefinisi. Coba AP_TEST_SET yang lain.")
else:
    cur = ap[(ap.model == MODEL) & (ap.split == SPLIT)].iloc[0]
    print(f"{MODEL} {SPLIT}%: AP = {cur.AP:.3f}, AP50 = {cur.AP50:.3f}")
    for metric in ("AP", "AP50"):
        print(f"\n{metric} (baris = model, kolom = % data latih)")
        display(ap.pivot(index="model", columns="split", values=metric).round(3))